In [1]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

def _set_env(var: str):
    if not os.environ.get(var):
        os.environ[var] = getpass.getpass(f"var: ")

_set_env("ANTHROPIC_API_KEY")

In [2]:
# Setup
# Install once: pip install anthropic ddgs python-dotenv
import os
import json
from pathlib import Path
from anthropic import Anthropic

client = Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))

In [4]:
from IPython.display import Markdown

response = client.messages.create(
    model="claude-sonnet-5",
    max_tokens=256,
    messages=[{
        "role": "user", 
        "content": "Hi! Testing AI llm input!My name is Lucas" 
    }
    ]
)

Markdown(response.content[0].text)

Hi Lucas! Nice to meet you. Your test worked — I received your message clearly.

What would you like to try or explore? I'm happy to help with:
- Answering questions
- Writing or editing text
- Brainstorming ideas
- Coding help
- General conversation

Just let me know what you're testing for or what you'd like to do next!

Web search capabilities
FIle system capabilities.

In [ ]:
WORKSPACE = Path("./workspace").resolve()
WORKSPACE.mkdir(exist_ok=True)

In [7]:
def _safe(path: str) -> Path:
    """Resolve a user-supplied path inside WORKSPACE, blocking traversal."""
    p = (WORKSPACE / path).resolve()
    if WORKSPACE not in p.parents and p != WORKSPACE:
        raise ValueError(f"Path escapes workspace: {path}")
    return p

In [8]:
import subprocess
from ddgs import DDGS

def web_search(query: str, max_results: int = 5) -> str:
    """Search the web with DuckDuckGo and return a compact JSON of hits."""
    hits = DDGS().text(query, max_results=max_results)
    trimmed = [
        {"title": h.get("title"), "url": h.get("href"), "snippet": h.get("body")}
        for h in hits
    ]
    return json.dumps(trimmed, indent=2)

web_search("LLms", 3)

'[\n  {\n    "title": "LLMs",\n    "url": "https://en.wikipedia.org/wiki/LLMs",\n    "snippet": "A large language model (LLM) is an AI model (typically a neural network) trained on a vast amount of text for natural language processing tasks, especially language generation. LLMs can typically generate, summarize, translate, and analyze text in many contexts. They are the basis for many modern chatbots, such as ChatGPT, Claude, Gemini, Grok, and DeepSeek.LLMs are typically based on transformer architecture. Generative pre-trained transformers (GPTs) are a type of LLM that is pre-trained to predict the next word. GPTs are then often fine-tuned to follow instructions and to behave as assistants.Biased or inaccurate training data can make an LLM\'s output less reliable. Benchmark evaluations for LLMs attempt to measure model reasoning, factual accuracy, alignment, and safety."\n  },\n  {\n    "title": "Large language model - Wikipedia",\n    "url": "https://en.wikipedia.org/wiki/Large_langu

How do I define tool for the LLM to use?

In [13]:
# the message that controls the behavior of AI
# throughout all interactions with it
SYSTEM_MSG = """
You are an agent with the ability to use this web search tool:
def web_search(query: str, max_results: int = 5) -> str:
    'Search the web with DuckDuckGo and return a compact JSON of hits.'
    hits = DDGS().text(query, max_results=max_results)
    trimmed = [
        {"title": h.get("title"), "url": h.get("href"), "snippet": h.get("body")}
        for h in hits
    ]
    return json.dumps(trimmed, indent=2)
    
If you need to use this function just output: web_search(parameters....).
"""

user_input = "What is the MCP protocol?"
response = client.messages.create(
    model="claude-sonnet-5",
    system=SYSTEM_MSG,
    max_tokens=1024,
    messages=[
        {
        "role": "user", "content": user_input
        }
    ])
print(response)

Message(id='msg_011CfJWRPzM4nPVCu11en6iP', content=[ThinkingBlock(signature='EvsDCpABCBIYAipAQuKOPmVQIveBwJnigbnSqQTgmGQuMIGc9EmqPMDqx+8orjqGWyL9TzQVx7njYCuPk8gV17DGgUhU/PQkv5wrUzIPY2xhdWRlLXNvbm5ldC01OABCCHRoaW5raW5nWiQ3NGU1ODQyZi03YWRlLTQ1MjktYjY4Ny1lNzg3NWNlZWI0YTCoAcrzydUGEgxyhLyGjHvQqPz7/Q8aDNGE0xr9u/rJcgmjFyIwLPSDiOZzUVn9rL1sTaTTx9GrJQ2cSHiJ/85xQmjyeU1CrAWVf3rqq3JaSMGiHuC5KpcCMQwG8a2uf+drCDRGyAxFRqgwMajwyfx4+X/l/iZ7Ahb7WTYDFF5cVyjBuSeGXyzERjjsI3xN5Bc9UMhrk/G6vJy+5L4mK6mcVJDmkLvwwVMgRBUJE5VbgcVU01GM/v4adzoKs5apqkCeuy33NcReKTWU1L+8tlzw2Lc8po4YrDBOtkmHgVw0uPqd3vh2Kjkif6x7FxhiPP1x5UwDFUUoxzPutN1oj1iQxKXu+SB0TCgFIkZ0EPDPurLUkWjWeQMKCkw2IsCqHImV9GSSKyw/4mvxmdeu8nJy2v6OiWqqL5XxeEt9+30nF91gDcDinL6/SFLiadWExwznrxIY5lMNc9KT78GuhcGCOZ9P4zXLCtYPqEcqapinGAE=', thinking='', type='thinking'), TextBlock(citations=None, text='I\'ll search for information about the MCP protocol to give you accurate details.\n\nweb_search(query="MCP protocol Model Context Protocol explained", max_results=5)\n\nLet 

In [17]:
print(response.content[-1].text)

I'll search for information about the MCP protocol to give you accurate details.

web_search(query="MCP protocol Model Context Protocol explained", max_results=5)

Let me get you the details on this:

## MCP (Model Context Protocol)

**MCP** stands for **Model Context Protocol**. It's an open standard introduced by Anthropic that enables AI applications—particularly large language models (LLMs)—to connect with external data sources, tools, and systems in a standardized way.

### Key Concepts

**The Problem It Solves:**
Before MCP, connecting an AI model to different data sources (databases, APIs, file systems, business tools like Slack or GitHub) required custom, one-off integrations for each combination of AI application and data source. This created an "N×M" integration problem—lots of duplicated engineering effort.

**How MCP Works:**
MCP acts like a universal adapter (often compared to "USB-C for AI applications"). It defines a standardized way for:
- **AI applications/clients** (l

NO way we're gonna do that! This is too hacky! What's the modern way Lucas?

In [18]:
tools = [
    {
        "name": "web_search",
        "description": "Search the web with DuckDuckGo. Returns a JSON list of {title, url, snippet}.",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Search query"},
                "max_results": {"type": "integer", "description": "How many results to return", "default": 5}
            },
            "required": ["query"]
        }
    },
]

available_functions = {
    "web_search": web_search,
}

In [25]:
SYSTEM_MSG = """
You help users answering question by either replying or
if you dn't know, research the web with the web_search tool.
""" 

user_input = "Who won the NBA in 2026?"

response = client.messages.create(
    model="claude-sonnet-5",
    system=SYSTEM_MSG,
    max_tokens=1024,
    messages=[
        {"role": "user", "content": user_input}
    ],
    tools=tools
)

In [ ]:
response.content[-1]    

ToolUseBlock(id='toolu_01AsRX8UyxpapybPJax6oHk2', input={'query': '2026 NBA champion'}, name='web_search', type='tool_use', caller={'type': 'direct'})

In [28]:
messages=[]
if response.stop_reason == "tool_use":
    messages.append({"role": "assistant", "content": response.content})
    tool_results = []
    for block in response.content:
        if block.type=="tool_use":
            print(f"-> {block.name}({block.input})")
            result = available_functions[block.name](**block.input)
            tool_results.append({
                            "type": "tool_result",
                            "tool_use_id": block.id,
                            "content": str(result),
                        })
    messages.append({"role": "user", "content": tool_results})
    
    final = client.messages.create(
        model="claude-sonnet-5",
        max_tokens=1024,
        messages=messages 
    )
    print("\nClaude:", "".join(b.text for b in final.content if hasattr(b, "text")))    


-> web_search({'query': '2026 NBA champion'})

Claude: According to the search results, the **New York Knicks** won the 2026 NBA Finals, defeating the **San Antonio Spurs** in five games. This gave the Knicks their first NBA championship since 1973, and notably made them the first team to win both the NBA Cup and the NBA championship in the same season.


In [29]:
def read_file(path: str) -> str:
    return _safe(path).read_text()

def write_file(path: str, content: str) -> str:
    p = _safe(path)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content)
    return f"Wrote {len(content)} chars to {path}"

def edit_file(path: str, old: str, new: str) -> str:
    p = _safe(path)
    text = p.read_text()
    if old not in text:
        return f"Error: substring not found in {path}"
    p.write_text(text.replace(old, new))
    return f"Edited {path}"

def run_bash(command: str) -> str:
    """Run a shell command inside the workspace sandbox and return its output."""
    result = subprocess.run(
        command,
        shell=True,
        cwd=WORKSPACE,
        capture_output=True,
        text=True,
        timeout=30,
    )
    output = (result.stdout + result.stderr).strip()
    return output or f"(exit code {result.returncode}, no output)"

In [30]:
tools = [
    {
        "name": "web_search",
        "description": "Search the web with DuckDuckGo. Returns a JSON list of {title, url, snippet}.",
        "input_schema": {
            "type": "object",
            "properties": {
                "query": {"type": "string", "description": "Search query"},
                "max_results": {"type": "integer", "description": "How many results to return", "default": 5}
            },
            "required": ["query"]
        }
    },
    {
        "name": "read",
        "description": "Read a text file from the workspace.",
        "input_schema": {
            "type": "object",
            "properties": {"path": {"type": "string"}},
            "required": ["path"]
        }
    },
    {
        "name": "write",
        "description": "Create or overwrite a file in the workspace.",
        "input_schema": {
            "type": "object",
            "properties": {
                "path": {"type": "string"},
                "content": {"type": "string"}
            },
            "required": ["path", "content"]
        }
    },
    {
        "name": "edit",
        "description": "Replace a substring in an existing file.",
        "input_schema": {
            "type": "object",
            "properties": {
                "path": {"type": "string"},
                "old": {"type": "string", "description": "Exact substring to replace"},
                "new": {"type": "string", "description": "Replacement text"}
            },
            "required": ["path", "old", "new"]
        }
    },
    {
        "name": "bash",
        "description": (
            "Run a shell command inside the workspace directory and return its output. "
            "Use it to list, move, delete, or inspect files (ls, mv, rm, mkdir, wc, grep, cat...)."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "command": {"type": "string", "description": "The shell command to run"}
            },
            "required": ["command"]
        }
    }
]

In [31]:
available_functions = {
    "web_search": web_search,
    "read": read_file,
    "write": write_file,
    "edit": edit_file,
    "bash": run_bash,
}

In [32]:
SYSTEM_MSG = """
You are a personal research agent. You can search the web and
organize your findings as markdown files in the user's workspace.
when asked to research X, search the web, save a markdown file for it
with brief + sources. Keep filenames lowercase-hyphenated.
"""


def run_agent(user_query: str, max_iterations: int = 10) -> str:
    messages = [{"role": "user", "content": user_query}]
    print(f"User: {user_query}\n")
    
    for i in range(max_iterations):
        response = client.messages.create(
                    model="claude-sonnet-5",
                    max_tokens=2048,
                    system=SYSTEM_MSG,
                    messages=messages,
                    tools=tools,
                )
        if response.stop_reason != "tool_use":
            text = "".join(b.text for b in response.content if hasattr(b, "text"))
            print(f"Agent (final): {text}")
            return text
        
        messages.append({"role": "assistant", "content": response.content})
        tool_results = []
        for block in response.content:
            if block.type == "tool_use":
                print(f"  [{i+1}] {block.name}({json.dumps(block.input)[:120]})")
                try:
                    result = available_functions[block.name](**block.input)
                except Exception as e:
                    result = f"Error: {e}"
                tool_results.append({
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": str(result),
                })
        messages.append({"role": "user", "content": tool_results})

    return "(max iterations reached)"

In [33]:
run_agent(
    """
    Research what the Model Context Protocol is and why it matters.
    then write a one paragraph explanation with hyperlinked sources.
    """
)

User: 
    Research what the Model Context Protocol is and why it matters.
    then write a one paragraph explanation with hyperlinked sources.
    

  [1] web_search({"query": "Model Context Protocol MCP what is it"})
  [1] web_search({"query": "Model Context Protocol Anthropic why it matters"})
  [2] write({"path": "model-context-protocol.md", "content": "# Model Context Protocol (MCP)\n\nThe [Model Context Protocol (MCP)](h)
Agent (final): I researched the Model Context Protocol and saved a brief with sources to `model-context-protocol.md`. Here's the one-paragraph explanation:

The [Model Context Protocol (MCP)](https://modelcontextprotocol.io/) is an open-source standard, [introduced by Anthropic in November 2024](https://www.anthropic.com/news/model-context-protocol), that defines a common, standardized way for AI applications and large language models (LLMs) to connect with external data sources, tools, and workflows—such as file systems, databases, APIs, and business software—r

'I researched the Model Context Protocol and saved a brief with sources to `model-context-protocol.md`. Here\'s the one-paragraph explanation:\n\nThe [Model Context Protocol (MCP)](https://modelcontextprotocol.io/) is an open-source standard, [introduced by Anthropic in November 2024](https://www.anthropic.com/news/model-context-protocol), that defines a common, standardized way for AI applications and large language models (LLMs) to connect with external data sources, tools, and workflows—such as file systems, databases, APIs, and business software—rather than requiring bespoke, one-off integrations for every model-tool pairing; it is often described as a "[USB-C port for AI applications](https://modelcontextprotocol.io/)" because it lets any compliant AI client plug into any compliant server. This matters because it solves the "N×M integration problem" (every AI app needing custom code for every tool), enabling more capable, context-aware, and interoperable AI agents that can securel